# Kaggle Smoke Test Script for OpenVLA (LIBERO-Object)

### 1. Clone OpenVLA Repository

This pulls down the official OpenVLA codebase. (Since your personal repo doesn't track this submodule yet, we pull it directly from the source).

In [ ]:
!git clone https://github.com/openvla/openvla

### 2. Install base OpenVLA requirements


In [ ]:
!pip install -r openvla/requirements-min.txt

### 3. Install manually discovered dependencies & enforce peft version constraint

In [ ]:
!pip install draccus accelerate datasets wandb peft==0.11.1 rich einops jsonlines matplotlib huggingface_hub bitsandbytes timm

### 4. Install TF data pipeline dependencies

⚠️ **Kaggle Risk**: May need to drop `==2.15.0` or use `--no-deps` if it conflicts with Kaggle's base TF.

In [ ]:
!pip install tensorflow==2.15.0 tensorflow_datasets==4.9.3 tensorflow_graphics==2021.12.3

### 5. Install dlimp from the custom GitHub fork

In [ ]:
!pip install git+https://github.com/moojink/dlimp_openvla

### 6. Download the dataset (now that huggingface-cli is guaranteed available)

In [ ]:
!mkdir -p datasets/openvla/modified_libero_rlds
!huggingface-cli download openvla/modified_libero_rlds \
    --repo-type dataset \
    --include "libero_object_no_noops/*" \
    --local-dir datasets/openvla/modified_libero_rlds

### 7. Disable W&B Interactive Prompt (prevents silent hanging on Kaggle)

In [ ]:
%env WANDB_MODE=disabled

### 8. Run the OpenVLA Finetuning Smoke Test (bf16)

In [ ]:
!torchrun --standalone --nnodes 1 --nproc-per-node 1 openvla/vla-scripts/finetune.py \
  --vla_path "openvla/openvla-7b" \
  --data_root_dir "datasets/openvla/modified_libero_rlds" \
  --dataset_name "libero_object_no_noops" \
  --run_root_dir "openvla_checkpoints" \
  --adapter_tmp_dir "openvla_checkpoints/tmp" \
  --use_lora True \
  --use_quantization False \
  --lora_rank 32 \
  --batch_size 2 \
  --grad_accumulation_steps 1 \
  --learning_rate 5e-4 \
  --image_aug False \
  --wandb_project "openvla-kaggle-smoke" \
  --wandb_entity "local-test" \
  --save_steps 10 \
  --max_steps 10

### 9. FALLBACK

If the standard bf16 run OOMs on the Kaggle GPU, comment out step 8 and run this cell instead.

In [ ]:
# !torchrun --standalone --nnodes 1 --nproc-per-node 1 openvla/vla-scripts/finetune.py \
#   --vla_path "openvla/openvla-7b" \
#   --data_root_dir "datasets/openvla/modified_libero_rlds" \
#   --dataset_name "libero_object_no_noops" \
#   --run_root_dir "openvla_checkpoints" \
#   --adapter_tmp_dir "openvla_checkpoints/tmp" \
#   --use_lora True \
#   --use_quantization True \
#   --lora_rank 32 \
#   --batch_size 2 \
#   --grad_accumulation_steps 1 \
#   --learning_rate 5e-4 \
#   --image_aug False \
#   --wandb_project "openvla-kaggle-smoke" \
#   --wandb_entity "local-test" \
#   --save_steps 10 \
#   --max_steps 10